<a href="https://colab.research.google.com/github/Danny3636/Generative-AI-Tasks/blob/main/Required_Task_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

From this task, I learned how different LLM prompting techniques such as Zero-shot, Chain-of-Thought, Role-Based Debate, and Structured Extraction can be applied to financial document analysis. The LLM judge demonstrated an effective method for evaluating these varied responses, providing insights into the strengths and weaknesses of each prompting strategy for answering specific business questions. Ultimately, this comprehensive approach allows for a robust, AI-assisted analysis of a company's financial strategy.

In [19]:
# Cell 1
!pip install google-generativeai -q

from google.colab import userdata


In [20]:
"""
Coinbase Q3 2024 Earnings Call — Prompt Engineering Analysis
=============================================================
Uses 4 different prompting techniques to answer:
  "Is Coinbase's strategy to reduce reliance on volatile trading fees
   actually working as of Q3 2024?"

Then an LLM Judge rates each response, and a final Executive Summary
is produced incorporating the judge's feedback.

FREE TO RUN — uses Google Gemini 2.5 Flash (free tier, no credit card).

Requirements:
    pip install google-generativeai

Setup:
    1. Go to https://aistudio.google.com/apikey
    2. Click "Create API key" (sign in with any Google account)
    3. Copy the key
    4. Set it: export GEMINI_API_KEY="your-key-here"
       OR in Colab: store as a Secret named GEMINI_API_KEY

Usage:
    python coinbase_prompt_analysis.py
"""

import google.generativeai as genai
import os
import time
import textwrap

# ── Gemini client setup ─────────────────────────────────────────────
# Try Colab secrets first, then env var
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = os.environ.get("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError(
        "No API key found!\n"
        "1. Get a FREE key at: https://aistudio.google.com/apikey\n"
        "2. In Colab: add it as a Secret named GEMINI_API_KEY\n"
        "   OR set env var: export GEMINI_API_KEY='your-key'"
    )

genai.configure(api_key=API_KEY)
MODEL_NAME = "gemini-2.5-flash"

# ── Transcript excerpt (key financial passages) ─────────────────────
TRANSCRIPT = textwrap.dedent("""\
COINBASE GLOBAL (COIN) — Q3 2024 EARNINGS CALL TRANSCRIPT (Oct 30 2024)
=======================================================================

BRIAN ARMSTRONG (CEO):
We had some softer market conditions in Q3, but overall, it was a really solid
quarter for Coinbase. It was our seventh consecutive quarter of positive adjusted
EBITDA. It was our fourth consecutive quarter of positive net income.

We've made a big effort to diversify our revenue over the years away from
transaction fee revenue, which is more volatile, it's not as predictable, it's
more market dependent. We've shifted more of that to subscription and services
revenue over time. And we've made incredible progress on that this year. So we're
now on pace to surpass $2 billion in subscription and services revenue in 2024.
That's up from just $1.4 billion in 2023.

This has resulted in a stronger balance sheet, which is now at $8.2 billion, and
that's given us a lot of financial flexibility. Our board has also authorized a
$1 billion stock buyback.

Stablecoin payments or transactions, we saw about $10 trillion of volume last
year. And we've already two-times'd that in 2024 — surpassed $20 trillion.
The market cap of USDC is up 45% year to date to $36 billion in Q3, up from
$25 billion at the start of this year.

Base is now the No. 1 Layer 2 solution out there — by transactions processed and
total value on the platform. Transactions increased 55% on Base quarter over
quarter.

ALESIA HAAS (CFO):
Q3 total revenue was $1.2 billion, our expenses were within outlook ranges, and
adjusted EBITDA was $449 million. Our balance sheet strengthened, and total USD
resources grew 5% quarter over quarter to $8.2 billion.

Total trading volume was $185 billion, down 18% quarter over quarter driven by
lower crypto asset volatility. Total transaction revenue was $573 million, down
27% quarter over quarter.

Subscription and services revenue was $556 million, down 7% quarter over quarter.
We were really pleased to grow native units in staking and custody. However, this
native unit growth was offset by lower average crypto asset prices.

Our Q3 stablecoin revenue grew 3%, as USDC market cap growth and our on-platform
USDC balance growth exceeded the impact of lower interest rates.

The number of Coinbase One paid subscribers continued to grow, and we saw all-time
highs in the third quarter.

We are expecting Q4 tech and dev and G&A expenses to be in the range of
$690 million to $730 million.

We're really pleased to see our long-term revenue diversification efforts paying
off as subscription and services revenue is on pace to exceed $2 billion this
year.

Our first financial goal is to continue to diversify our revenue; second,
maintaining expense discipline; and third, generating positive adjusted EBITDA in
all market conditions.

From the Q&A:
- Coinbase holds ~$1.3 billion in crypto investments (25% of net cash balance).
- Consumer stablecoin pair trading volume grew significantly Q/Q.
- Derivatives revenue showing early but promising growth.
- No material changes to fee rate structure; blended average change driven by
  mix shift toward stable pair trading and lower nontrading transaction revenue.
- USDC was the fastest-growing major stablecoin in Q3 and YTD.
""")

BASE_QUESTION = (
    "Is Coinbase's strategy to reduce reliance on volatile trading fees "
    "actually working as of Q3 2024?"
)


# ── Helper: call Gemini ─────────────────────────────────────────────
def ask_gemini(system: str, user: str) -> str:
    """Send a message to Gemini and return the text response."""
    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        system_instruction=system,
    )
    # Respect free-tier rate limits (15 RPM for Flash)
    time.sleep(5)
    response = model.generate_content(user)
    return response.text


# ====================================================================
# 1 ▸ FOUR PROMPTING TECHNIQUES
# ====================================================================

def technique_zero_shot() -> str:
    """Technique 1 — Zero-Shot: plain question, no examples or scaffolding."""
    print("\n⏳  Running Technique 1: Zero-Shot ...")
    system = "You are a senior equity research analyst."
    user = f"""Read the following earnings-call transcript excerpt, then answer
the question.

TRANSCRIPT:
{TRANSCRIPT}

QUESTION: {BASE_QUESTION}
"""
    return ask_gemini(system, user)


def technique_chain_of_thought() -> str:
    """Technique 2 — Chain-of-Thought: forces step-by-step reasoning."""
    print("⏳  Running Technique 2: Chain-of-Thought ...")
    system = "You are a senior equity research analyst."
    user = f"""Read the following earnings-call transcript excerpt.

TRANSCRIPT:
{TRANSCRIPT}

QUESTION: {BASE_QUESTION}

Think step by step. For EACH step, cite the specific numbers from the
transcript that support your reasoning.

Step 1 — Identify the revenue diversification metrics mentioned.
Step 2 — Compare transaction revenue vs. subscription/services revenue trends.
Step 3 — Evaluate whether the non-trading revenue streams are growing fast
          enough to offset trading-fee volatility.
Step 4 — Assess management's own framing and any caveats.
Step 5 — Deliver a clear verdict with confidence level (high / medium / low).
"""
    return ask_gemini(system, user)


def technique_role_debate() -> str:
    """Technique 3 — Role-Based Debate: bull vs. bear, then synthesis."""
    print("⏳  Running Technique 3: Role-Based Debate ...")
    system = (
        "You will adopt two personas sequentially — a Bull Analyst and a Bear "
        "Analyst — then synthesize their arguments into a balanced verdict."
    )
    user = f"""TRANSCRIPT:
{TRANSCRIPT}

QUESTION: {BASE_QUESTION}

Instructions — produce THREE clearly labeled sections:

## BULL CASE
Argue that the diversification strategy IS working. Use specific numbers.

## BEAR CASE
Argue that the strategy is NOT (yet) working or has significant risks. Use
specific numbers and highlight what could go wrong.

## SYNTHESIS
Weigh both sides and deliver a balanced, evidence-based verdict.
"""
    return ask_gemini(system, user)


def technique_structured_extraction() -> str:
    """Technique 4 — Structured Extraction → Analysis: extract data first,
    then reason over it."""
    print("⏳  Running Technique 4: Structured Extraction → Analysis ...")

    # Step A — extract structured data
    system_extract = (
        "You are a financial data extraction engine. "
        "Respond ONLY in valid JSON. No markdown fences, no preamble."
    )
    user_extract = f"""From the transcript below, extract the following fields
into a JSON object. Use null if a value is not stated.

Fields:
- q3_total_revenue
- q3_transaction_revenue
- q3_transaction_revenue_qoq_change
- q3_sub_services_revenue
- q3_sub_services_revenue_qoq_change
- annualized_sub_services_2024_pace
- sub_services_2023
- q3_adjusted_ebitda
- consecutive_positive_ebitda_quarters
- consecutive_positive_net_income_quarters
- q3_trading_volume
- q3_trading_volume_qoq_change
- usdc_market_cap_q3
- usdc_ytd_growth
- base_txn_qoq_growth
- stablecoin_revenue_qoq_change
- balance_sheet_usd_resources
- key_growth_drivers (list of strings)
- key_risks_or_headwinds (list of strings)

TRANSCRIPT:
{TRANSCRIPT}
"""
    raw_json = ask_gemini(system_extract, user_extract)

    # Step B — analyse the structured data
    system_analyse = "You are a senior equity research analyst."
    user_analyse = f"""Below is structured data extracted from Coinbase's Q3 2024
earnings call, followed by the original question.

EXTRACTED DATA:
{raw_json}

QUESTION: {BASE_QUESTION}

Using the extracted data, write a concise, evidence-rich analysis (≤ 300 words).
End with a clear YES / PARTIALLY / NO verdict and a one-sentence rationale.
"""
    analysis = ask_gemini(system_analyse, user_analyse)
    return f"### Extracted JSON\n```json\n{raw_json}\n```\n\n### Analysis\n{analysis}"


# ====================================================================
# 2 ▸ LLM JUDGE
# ====================================================================

def judge_responses(responses: dict[str, str]) -> str:
    """Use Gemini as a judge to score and critique each technique's output."""
    print("\n⚖️  Running LLM Judge ...")

    entries = "\n\n".join(
        f"=== RESPONSE FROM: {name} ===\n{text}"
        for name, text in responses.items()
    )

    system = (
        "You are an impartial evaluation judge for financial analysis quality. "
        "Be rigorous and specific in your scoring."
    )
    user = f"""Below are four analyst responses to the same question about
Coinbase's Q3 2024 earnings. Each was generated using a different prompting
technique.

QUESTION: {BASE_QUESTION}

{entries}

Score each response on these five criteria (1-10 scale):

1. **Accuracy** — Are the cited numbers correct and faithfully drawn from
   the transcript?
2. **Depth** — Does the response go beyond surface-level observations?
3. **Structure** — Is the response well-organized and easy to follow?
4. **Nuance** — Does it acknowledge trade-offs, risks, or caveats?
5. **Actionability** — Could an investor make a decision based on this?

For each response, provide:
- A score table (criteria + score)
- A TOTAL score (sum of 5 criteria, max 50)
- 2-3 sentences of qualitative feedback

Finally, rank the four techniques from best to worst and explain why.
"""
    return ask_gemini(system, user)


# ====================================================================
# 3 ▸ FINAL EXECUTIVE SUMMARY
# ====================================================================

def executive_summary(responses: dict[str, str], judge_output: str) -> str:
    """Produce a 200-word executive summary informed by the judge."""
    print("\n📝  Generating Executive Summary ...")
    system = "You are a senior equity research analyst writing for a C-suite audience."
    user = f"""You have access to four analyst responses AND a judge's evaluation
of those responses.

JUDGE'S EVALUATION:
{judge_output}

ANALYST RESPONSES:
{chr(10).join(f"--- {k} ---{chr(10)}{v}" for k, v in responses.items())}

TASK:
Write a single, polished **200-word Executive Summary** that answers:
"{BASE_QUESTION}"

Requirements:
- Incorporate the strongest evidence identified by the judge.
- Acknowledge the most important caveat the judge highlighted.
- Include at least 3 specific numbers from the transcript.
- End with a one-sentence investment implication.
- EXACTLY ~200 words (±20).
"""
    return ask_gemini(system, user)


# ====================================================================
# MAIN
# ====================================================================

def main():
    print("=" * 70)
    print("  COINBASE Q3 2024 — PROMPT ENGINEERING ANALYSIS PIPELINE")
    print("  Model: Google Gemini 2.5 Flash (FREE tier)")
    print("=" * 70)

    # ── Step 1: Run four techniques ─────────────────────────────────
    responses = {
        "1. Zero-Shot":                technique_zero_shot(),
        "2. Chain-of-Thought":         technique_chain_of_thought(),
        "3. Role-Based Debate":        technique_role_debate(),
        "4. Structured Extraction":    technique_structured_extraction(),
    }

    for name, text in responses.items():
        print(f"\n{'─' * 60}")
        print(f"  {name}")
        print(f"{'─' * 60}")
        print(text[:2000], "..." if len(text) > 2000 else "")

    # ── Step 2: LLM Judge ───────────────────────────────────────────
    judge_output = judge_responses(responses)
    print(f"\n{'━' * 60}")
    print("  ⚖️  JUDGE'S EVALUATION")
    print(f"{'━' * 60}")
    print(judge_output)

    # ── Step 3: Executive Summary ───────────────────────────────────
    summary = executive_summary(responses, judge_output)
    print(f"\n{'━' * 60}")
    print("  📝  FINAL EXECUTIVE SUMMARY")
    print(f"{'━' * 60}")
    print(summary)

    # ── Save everything to file ─────────────────────────────────────
    with open("coinbase_analysis_results.md", "w") as f:
        f.write("# Coinbase Q3 2024 — Prompt Engineering Analysis Results\n\n")
        f.write(f"**Model:** {MODEL_NAME} (Gemini Free Tier)\n\n")
        for name, text in responses.items():
            f.write(f"## {name}\n\n{text}\n\n---\n\n")
        f.write(f"## Judge's Evaluation\n\n{judge_output}\n\n---\n\n")
        f.write(f"## Executive Summary\n\n{summary}\n")

    print("\n✅  Full results saved to: coinbase_analysis_results.md")


if __name__ == "__main__":
    main()


  COINBASE Q3 2024 — PROMPT ENGINEERING ANALYSIS PIPELINE
  Model: Google Gemini 2.5 Flash (FREE tier)

⏳  Running Technique 1: Zero-Shot ...
⏳  Running Technique 2: Chain-of-Thought ...
⏳  Running Technique 3: Role-Based Debate ...
⏳  Running Technique 4: Structured Extraction → Analysis ...

────────────────────────────────────────────────────────────
  1. Zero-Shot
────────────────────────────────────────────────────────────
Yes, Coinbase's strategy to reduce reliance on volatile trading fees appears to be **largely working and making significant progress as of Q3 2024**, though it's important to note that full decoupling from crypto market conditions is an ongoing effort.

Here's the evidence supporting this conclusion:

**Evidence of the Strategy Working:**

1.  **Revenue Mix Shift:**
    *   CEO Brian Armstrong explicitly states, "We've made a big effort to diversify our revenue over the years away from transaction fee revenue, which is more volatile... We've shifted more of that